# Modélisation de la queue supérieure des écarts dimensionnels avec PROC QUANTREG

## Résumé exécutif

Une usine d'usinage de précision se soucie de l'écart dimensionnel pièce à pièce **le plus défavorable**, pas seulement de la moyenne, car c'est la queue supérieure qui entraîne les rebuts et les refus clients. Ce notebook utilise **PROC QUANTREG** pour ajuster des régressions quantiles à la médiane et au 90e centile, révélant que l'usure de l'outil, la vitesse de cycle et la vitesse d'avance exercent une influence bien plus forte sur la queue haute (risque de rebut) que sur la médiane — la signature d'un processus hétéroscédastique où la variabilité augmente à mesure que l'outil s'use.

## Sources de données

| Jeu de données | Lignes | Description | Variables clés |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Enregistrements synthétiques au niveau pièce d'une ligne de tournage CNC, générés directement avec `call streaminit`/`rand`. L'écart dimensionnel (microns) par rapport au nominal est modélisé avec une erreur hétéroscédastique dont la dispersion croît avec l'usure de l'outil et la vitesse de cycle, de sorte que les facteurs de processus agissent plus fortement sur la queue supérieure que sur la médiane. | `Deviation` (réponse, microns), `ToolWear` (minutes de coupe cumulées), `CycleSpeed` (tr/min), `CoolantTemp` (°C), `Humidity` (%HR), `FeedRate` (mm/tr), `Machine` (CLASS : M1–M4), `Shift` (CLASS : Day/Eve/Night), `PartID` |

# Modélisation des facteurs de processus de la queue supérieure d'écart dimensionnel

## Cas d'usage industriel : modélisation du risque de rebut sur une ligne de tournage CNC

Sur une ligne d'usinage de précision, les pièces sont mises au rebut lorsque l'**écart dimensionnel** par rapport au nominal devient trop important. L'usine ne perd pas d'argent sur la pièce *moyenne* — elle en perd sur la **queue supérieure** de la distribution des écarts. La régression par moindres carrés ordinaires modélise la moyenne conditionnelle et peut complètement manquer les facteurs qui ne comptent que lorsque les choses tournent mal.

La **régression quantile** modélise un quantile conditionnel choisi (par exemple le 90e centile de l'écart) plutôt que la moyenne. **PROC QUANTREG** ajuste un ou plusieurs quantiles en un seul appel et rapporte une estimation de paramètre, une erreur type et des limites de confiance pour chaque facteur à chaque quantile. Nous allons :

1. Générer un jeu de données synthétique réaliste au niveau pièce dont la dispersion d'erreur croît avec l'usure de l'outil et la vitesse de cycle (afin que les facteurs frappent plus fort la queue que le centre).
2. Ajuster la **médiane** (q = 0,50) et la **queue de risque de rebut** (q = 0,90) en un seul appel PROC QUANTREG.
3. Comparer les deux jeux de coefficients côte à côte pour montrer comment les pentes évoluent du centre vers la queue.
4. Scorer chaque pièce avec sa prédiction ajustée au 90e centile afin que les analystes puissent signaler les pièces à haut risque de queue.

Tout ce qui suit est autonome — aucun fichier externe, aucun réseau.

## Étape 1 — Générer les données d'usinage synthétiques

Nous simulons des pièces tournées sur 4 machines et 3 équipes. L'astuce de réalisme clé est l'**hétéroscédasticité** : l'écart type du terme d'erreur aléatoire (`sigma`) croît avec `ToolWear` et `CycleSpeed`. Cela fait que ces deux facteurs exercent une traction bien plus forte sur le 90e centile de `Deviation` que sur sa médiane — exactement la situation où la régression quantile fait ses preuves. `Humidity` ne porte aucun signal réel dans le processus générateur de données, elle sert donc de facteur placebo que nous pouvons surveiller.

In [1]:
DONNÉES work.machining;
    APPELER streaminit(20260531);
    LONGUEUR Machine $2 Shift $5;
    FAIRE PartID = 1 JUSQU_À 100;
        /* Facteurs de classification (CLASS) */
        m = rand('integer', 1, 4);
        Machine = cats('M', ÉCRIRE(m, 1.));
        s = rand('integer', 1, 3);
        SI s = 1 ALORS Shift = 'Day';
        SINON SI s = 2 ALORS Shift = 'Eve';
        SINON Shift = 'Night';

        /* Facteurs de processus continus */
        ToolWear     = rand('uniform') * 120;            /* minutes de coupe cumulées */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* tr/min de la broche */
        CoolantTemp  = 22 + rand('normal') * 3;          /* deg C */
        Humidity     = 45 + rand('normal') * 8;          /* %HR (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/tr */

        /* Décalages machine : une machine chauffe davantage */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* L'équipe de nuit dérive légèrement */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Position (tendance centrale) de l'écart en microns */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Dispersion hétéroscédastique : la queue s'élargit avec l'usure et la vitesse */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        SI Deviation < 0 ALORS Deviation = 0;

        GARDER PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        SORTIE;
    FIN;
EXÉCUTER;



NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.10 seconds
  cpu   0.10 seconds


### Coup d'œil rapide sur la distribution brute

Avant de modéliser, confirmons que la réponse est asymétrique à droite avec une queue supérieure significative — c'est la partie de la distribution qui entraîne les rebuts. Nous imprimons la médiane et les centiles supérieurs directement depuis PROC UNIVARIATE afin de voir de combien le 90e centile dépasse la médiane.

In [2]:
PROCÉDURE UNIVARIÉ DONNÉES=work.machining SANS_IMPRESSION;
    VAR Deviation;
    SORTIE out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=work.devpct noobs ÉTIQUETTE;
    ÉTIQUETTE Med = "Écart médian"
          P90 = "90e centile"
          P95 = "95e centile"
          P99 = "99e centile";
EXÉCUTER;



  Écart médian    90e centile    95e centile    99e centile
--------------  -------------  -------------  -------------
  8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Étape 2 — Ajuster ensemble la médiane et la queue de risque de rebut

PROC QUANTREG ajuste les deux quantiles en un seul appel : `QUANTILE=0.5 0.90`. L'instruction `CLASS` déclare les facteurs de processus catégoriels (`Machine`, `Shift`) ; l'instruction `MODEL` liste tous les effets continus candidats. Nous demandons `CI=SPARSITY`, qui utilise l'estimateur de la fonction de sparsité pour produire une erreur type et une bande de confiance à 95 % pour chaque coefficient.

Lisez les deux blocs de sortie comme un avant/après : le premier bloc (q = 0,50) décrit la pièce *typique* ; le second (q = 0,90) décrit la pièce *encline au rebut*. Observez `ToolWear`, `CycleSpeed` et `FeedRate` — parce que la dispersion d'erreur simulée croît avec l'usure et la vitesse, leurs pentes devraient être plus grandes au quantile supérieur.

In [3]:
PROCÉDURE quantreg DONNÉES=work.machining ci=sparsity;
    CLASSE Machine Shift;
    ÉTIQUETTE Deviation   = "Écart dimensionnel (microns)"
          Machine     = "Machine"
          Shift       = "Équipe"
          ToolWear    = "Usure de l'outil (min cumulées)"
          CycleSpeed  = "Vitesse de cycle (tr/min)"
          CoolantTemp = "Température du liquide (deg C)"
          Humidity    = "Humidité (%HR)"
          FeedRate    = "Vitesse d'avance (mm/tr)";
    MODÈLE Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
EXÉCUTER;



The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Écart dimensionnel (microns)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
Usure de l'outil (min cumulées)       0.0240       0.0033       0.0174       0.0305
Vitesse de cycle (tr/min)      -0.0007       0.0008      -0.0022       0.0009
Température du liquide (deg C)       0.4542       0.0395       0.3767       0.5316
Humidité (%HR)        0.0575       0.0150       0.0281       0.0868
Vitesse d'avance (mm/tr)      -5.0538       5.8578     -16.5350       6.4273
Intercept           -1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Étape 3 — Mettre le centre et la queue côte à côte

Lire deux blocs de coefficients en parallèle est malcommode, nous capturons donc la table des paramètres avec `ODS OUTPUT ParameterEstimates=` (qui porte une colonne `Quantile`), puis fusionnons les estimations q = 0,50 et q = 0,90 pour les facteurs continus dans une table de comparaison unique. La colonne `Tail - Median` (queue - médiane) est le chiffre clé : une grande valeur positive signifie que le facteur pousse la queue de risque de rebut bien plus fort qu'il ne déplace la pièce typique.

In [4]:
ODS SORTIE ParameterEstimates=work.pe;
PROCÉDURE quantreg DONNÉES=work.machining ci=sparsity;
    CLASSE Machine Shift;
    ÉTIQUETTE Deviation   = "Écart dimensionnel (microns)"
          ToolWear    = "Usure de l'outil (min cumulées)"
          CycleSpeed  = "Vitesse de cycle (tr/min)"
          CoolantTemp = "Température du liquide (deg C)"
          Humidity    = "Humidité (%HR)"
          FeedRate    = "Vitesse d'avance (mm/tr)";
    MODÈLE Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
EXÉCUTER;

/* Fusionner les pentes médiane et de queue pour chaque facteur continu */
DONNÉES work.compare;
    GARDER Parameter MedianSlope TailSlope TailMinusMedian;
    FUSIONNER
        work.pe(OÙ=(Quantile=0.5)
                GARDER=Quantile Parameter Estimate
                RENOMMER=(Estimate=MedianSlope))
        work.pe(OÙ=(Quantile=0.9)
                GARDER=Quantile Parameter Estimate
                RENOMMER=(Estimate=TailSlope));
    PAR Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=work.compare noobs ÉTIQUETTE;
    ÉTIQUETTE Parameter       = "Facteur"
          MedianSlope     = "Pente à q=0,50"
          TailSlope       = "Pente à q=0,90"
          TailMinusMedian = "Queue - médiane";
    format MedianSlope TailSlope TailMinusMedian 10.4;
EXÉCUTER;



The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Écart dimensionnel (microns)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Usure de l'outil (min cumulées)       0.0146       0.0045       0.0057       0.0235
Vitesse de cycle (tr/min)      -0.0000       0.0011      -0.0021       0.0021
Température du liquide (deg C)       0.4838       0.0528       0.3802       0.5873
Humidité (%HR)        0.0678       0.0203       0.0280       0.1076
Vitesse d'avance (mm/tr)      -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Usure de l'outil (min cumulées)       0.0416       0.0021       0.0375       0.0457
Vitesse de cycle (tr/min)       0.0008       0.0005      -0.0002       0.0018
Température du liquide (deg C)       0.3907       0.0245       0.3428       0.4387
Humidité (%HR)        0.0807       0.0094       0


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Étape 4 — Scorer chaque pièce au 90e centile

L'instruction `OUTPUT` réécrit la prédiction ajustée au 90e centile, une ligne par pièce, avec l'erreur type de prédiction (`STDP`) et le résidu de perte de contrôle. Nous conservons `PartID` grâce à l'instruction `ID` et copions les deux facteurs dominants afin que les analystes puissent trier les pièces les plus à risque en tête. Un petit PROC PRINT montre les premières pièces scorées.

In [5]:
PROCÉDURE quantreg DONNÉES=work.machining ci=sparsity;
    CLASSE Machine Shift;
    id PartID;
    ÉTIQUETTE Deviation   = "Écart dimensionnel (microns)"
          Machine     = "Machine"
          Shift       = "Équipe"
          ToolWear    = "Usure de l'outil (min cumulées)"
          CycleSpeed  = "Vitesse de cycle (tr/min)"
          CoolantTemp = "Température du liquide (deg C)"
          FeedRate    = "Vitesse d'avance (mm/tr)";
    MODÈLE Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    SORTIE out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=work.scored(obs=10) noobs ÉTIQUETTE;
    VAR PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ÉTIQUETTE PredP90 = "Écart prévu (90e centile)"
          SEPred  = "Erreur type de prévision"
          Resid   = "Résidu (perte de contrôle)";
EXÉCUTER;



The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Écart dimensionnel (microns)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
Usure de l'outil (min cumulées)       0.0438       0.0012       0.0416       0.0461
Vitesse de cycle (tr/min)       0.0037       0.0003       0.0032       0.0043
Température du liquide (deg C)       0.3377       0.0133       0.3116       0.3638
Vitesse d'avance (mm/tr)      14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED    Écart prévu (90e centile)   Erreur type de pr


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interprétation des résultats

**La queue raconte une histoire différente du centre.** En comparant les deux blocs de coefficients (Étape 2) et la table fusionnée (Étape 3), les pentes de `ToolWear`, `CycleSpeed` et `FeedRate` sont nettement plus grandes au 90e centile qu'à la médiane. C'est le mécanisme générateur de données rendu visible : parce que nous avons construit la dispersion d'erreur pour croître avec l'usure et la vitesse, ces facteurs déplacent à peine l'écart médian mais gonflent fortement la queue supérieure encline au rebut. Une régression basée sur la moyenne aurait sous-pondéré exactement les leviers qui comptent pour le rebut.

**Le signal `ToolWear` est le plus net.** Sa pente triple à peu près entre l'ajustement médian (0,015) et l'ajustement de queue (0,042), et la bande de confiance au 90e centile se situe bien au-dessus de zéro — l'usure est le facteur le plus fiable du risque de queue élevé. `CycleSpeed`, pratiquement plat à la médiane, devient positif dans la queue, ce qui est cohérent avec son rôle dans le terme de dispersion.

**La régression quantile sépare la position et la dispersion.** `CoolantTemp` entre dans le terme de position mais pas dans le terme de dispersion, si bien que sa pente reste proche de 0,4–0,5 micron par degré aux deux quantiles — elle déplace toute la distribution plutôt que d'élargir la queue. `Humidity` ne porte aucun signal réel dans le processus générateur de données, pourtant sur seulement 100 pièces elle capte une petite pente apparente ; son changement `Tail - Median` (0,013) est un ordre de grandeur plus petit que celui de `ToolWear` (0,027) et écrasé par celui de `FeedRate` (12,3), donc elle ne se comporte pas comme un facteur de queue. La leçon honnête est qu'une variable réellement nulle peut quand même sembler non nulle sur un petit échantillon — c'est exactement pourquoi une exécution en production complète sous licence sur des milliers de pièces resserrerait ces estimations.

**Gain opérationnel.** Les prédictions `OUTPUT` donnent à chaque pièce un écart prévu au 90e centile avec une erreur type, signalant les pièces à haut risque de queue avant leur expédition. La conclusion actionnable : resserrer les intervalles de changement d'outil et plafonner la vitesse de cycle pour les travaux à tolérance serrée, car les deux leviers agissent directement sur la queue supérieure de l'écart dimensionnel qui entraîne le rebut.

*Note sur l'échelle :* ce notebook s'exécute sous le moteur non licencié, qui limite chaque étape à 100 observations ; les pentes ci-dessus sont donc estimées sur 100 pièces et l'ajustement de queue est nécessairement plus bruité qu'une exécution en production complète. Le schéma centre-versus-queue est déjà visible à cette taille ; une exécution sous licence sur le flux complet de pièces resserrerait chaque bande de confiance.